# Integrated Figure 1 Four-Panel Plot

This notebook redraws the four manuscript panels from raw data with one shared style system:

- Task 1 learning curve
- Task 1 best-score distribution
- Task 2 learning curve
- Task 2 best-score distribution

Attached/reference images are used only as visual references; all plotted values are recomputed from source data.

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "figure1" / "code"))

import matplotlib
matplotlib.use('Agg')  # prevents external plot windows during scripted runs
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import FixedLocator, FormatStrFormatter
import numpy as np
import pandas as pd

from publication_plot_style import TASK1_COLORS, TASK1_LABELS, TASK2_COLORS, TASK2_LABELS

DATA_DIR = Path('data')
OUTPUT_DIR = Path('figure1/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK1_DATA_PATH = DATA_DIR / 'choice_category_uniform_105.csv'
TASK2_DATA_PATH = DATA_DIR / 'summary_data_0723_task2.csv'

STYLE = {
    'learning_figsize': (4.6, 3.25),
    'distribution_figsize': (4.6, 3.25),
    'combined_figsize': (17.8, 3.45),
    'dpi': 300,
    'axis_width': 1.8,
    'tick_width': 1.6,
    'tick_length': 5.8,
    'line_width': 2.2,
    'box_line_width': 1.6,
    'median_width': 2.0,
    'label_size': 17,
    'tick_size': 14.5,
    'legend_size': 10.5,
    'panel_label_size': 13,
    'point_size': 22,
    'point_alpha': 0.88,
    'point_jitter': 0.060,
    'box_width': 0.24,
    'violin_width': 0.24,
    'scatter_offset': -0.24,
    'box_offset': 0.0,
    'violin_offset': 0.145,
    'group_step': 0.82,
    'ribbon_alpha': 0.25,
    'learning_y_ticks': [45, 60, 75, 90],
    'task1_learning_y_ticks': [50, 65, 80, 95],
    'task2_learning_y_ticks': [45, 60, 75, 90],
    'task1_learning_ylim': (50, 95),
    'task2_learning_ylim': (45, 90),
    'learning_xlim': (-0.06, 1.0),
    'distribution_group_slots': 3,
    'distribution_start': 1.08,
    'distribution_left_margin': 0.42,
    'score_y_ticks': [70, 80, 90, 100],
    'score_ylim': (70, 102),
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.grid': False,
})

In [2]:
def lighten_color(color, amount=0.25):
    rgb = np.array(mcolors.to_rgb(color))
    return tuple((1 - amount) * rgb + amount)


def centered_rolling_mean(values, window=5):
    return (
        pd.Series(np.asarray(values, dtype=float))
        .rolling(window=window, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )


def normalize_progress(trial_values, min_trial, max_trial):
    trial_values = np.asarray(trial_values, dtype=float)
    if max_trial <= min_trial:
        return np.zeros_like(trial_values, dtype=float)
    return (trial_values - min_trial) / (max_trial - min_trial)


def extend_after_first_perfect(df, *, subject_col, group_col, trial_col, score_col, max_trial_by_group):
    df = df.copy()
    perfect_score = df[score_col].max()
    extended_rows = []

    for _, subject_df in df.groupby([subject_col, group_col], dropna=False):
        subject_df = subject_df.sort_values(trial_col).copy()
        if subject_df.empty:
            continue

        group_value = subject_df[group_col].iloc[0]
        max_trial = int(max_trial_by_group[group_value])
        subject_df = subject_df[subject_df[trial_col] <= max_trial].copy()
        existing_trials = set(subject_df[trial_col].astype(int).tolist())
        subject_parts = [subject_df]

        perfect_hits = subject_df.loc[subject_df[score_col] >= perfect_score, trial_col]
        if not perfect_hits.empty:
            first_perfect_trial = int(perfect_hits.min())
            fill_trials = [
                trial for trial in range(first_perfect_trial + 1, max_trial + 1)
                if trial not in existing_trials
            ]
            if fill_trials:
                fill_df = pd.DataFrame({
                    subject_col: subject_df[subject_col].iloc[0],
                    group_col: group_value,
                    trial_col: fill_trials,
                    score_col: perfect_score,
                })
                subject_parts.append(fill_df)

        extended_rows.append(pd.concat(subject_parts, ignore_index=True))

    if not extended_rows:
        raise ValueError('No data available after extending perfect-score trials.')
    return pd.concat(extended_rows, ignore_index=True)


def compute_learning_curves(df, *, subject_col, group_col, trial_col, score_col, group_order, max_trial_by_group):
    extended_df = extend_after_first_perfect(
        df,
        subject_col=subject_col,
        group_col=group_col,
        trial_col=trial_col,
        score_col=score_col,
        max_trial_by_group=max_trial_by_group,
    )

    curve_dict = {}
    for group in group_order:
        sub = extended_df.loc[extended_df[group_col] == group].copy()
        if sub.empty:
            curve_dict[group] = pd.DataFrame()
            continue

        stats = (
            sub.groupby(trial_col)[score_col]
            .agg(['mean', 'std', 'count'])
            .reset_index()
            .sort_values(trial_col)
        )
        stats['sem'] = (stats['std'] / np.sqrt(stats['count'])).fillna(0)
        stats['x'] = normalize_progress(stats[trial_col], 1, max_trial_by_group[group])
        stats['mean_smooth'] = centered_rolling_mean(stats['mean'], window=5)
        stats['sem_smooth'] = centered_rolling_mean(stats['sem'], window=5)
        curve_dict[group] = stats

    return curve_dict


def style_panel_axis(ax, *, xlabel='', ylabel='', x_ticks=None, y_ticks=None, xlim=None, ylim=None, x_tick_format=None):
    ax.set_title('')
    ax.set_xlabel(xlabel, fontsize=STYLE['label_size'], fontweight='bold', labelpad=6)
    ax.set_ylabel(ylabel, fontsize=STYLE['label_size'], fontweight='bold', labelpad=6)

    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if x_ticks is not None:
        ax.xaxis.set_major_locator(FixedLocator(x_ticks))
    if y_ticks is not None:
        ax.yaxis.set_major_locator(FixedLocator(y_ticks))
    if x_tick_format is not None:
        ax.xaxis.set_major_formatter(FormatStrFormatter(x_tick_format))

    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(STYLE['axis_width'])
    ax.spines['bottom'].set_linewidth(STYLE['axis_width'])
    ax.spines['bottom'].set_position(('outward', 6))

    if y_ticks is not None and len(y_ticks) >= 2:
        ax.spines['left'].set_bounds(y_ticks[0], y_ticks[-1])
    if x_ticks is not None and len(x_ticks) >= 2:
        ax.spines['bottom'].set_bounds(x_ticks[0], x_ticks[-1])

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=STYLE['tick_size'],
        direction='out',
        length=STYLE['tick_length'],
        width=STYLE['tick_width'],
        top=False,
        right=False,
        pad=4,
    )


def plot_learning_panel(ax, curve_dict, *, group_order, colors, labels, ylim, y_ticks=None, show_legend=True):
    if y_ticks is None:
        y_ticks = STYLE['learning_y_ticks']

    for group in group_order:
        curve = curve_dict[group]
        if curve.empty:
            continue

        x = curve['x'].to_numpy(dtype=float)
        y = curve['mean_smooth'].to_numpy(dtype=float)
        sem = curve['sem_smooth'].to_numpy(dtype=float)
        color = colors[group]

        ax.plot(x, y, color=color, linewidth=STYLE['line_width'], label=labels[group], zorder=3)
        ax.fill_between(
            x,
            y - sem,
            y + sem,
            color=color,
            alpha=STYLE['ribbon_alpha'],
            linewidth=0,
            zorder=2,
        )

    style_panel_axis(
        ax,
        xlabel='Trial Progression',
        ylabel='Mean Score',
        x_ticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        y_ticks=y_ticks,
        xlim=STYLE['learning_xlim'],
        ylim=ylim,
        x_tick_format='%.1f',
    )

    if show_legend:
        legend = ax.legend(
            frameon=False,
            fontsize=STYLE['legend_size'],
            loc='lower right',
            handlelength=2.0,
            borderaxespad=0.2,
        )
        for line in legend.get_lines():
            line.set_linewidth(STYLE['line_width'])


def plot_raincloud_panel(
    ax,
    df,
    *,
    group_col,
    score_col,
    group_order,
    colors,
    labels,
    ylabel='Best score',
    show_violin=True,
    show_scatter=False,
    group_alignment='center',
):
    data = [df.loc[df[group_col] == group, score_col].dropna().to_numpy(dtype=float) for group in group_order]
    slot_count = max(len(group_order), STYLE['distribution_group_slots'])
    slot_span = (slot_count - 1) * STYLE['group_step']
    if group_alignment == 'left':
        start = STYLE['distribution_start']
    else:
        start = STYLE['distribution_start'] + (slot_count - len(group_order)) * STYLE['group_step'] / 2
    group_centers = np.arange(len(group_order)) * STYLE['group_step'] + start
    scatter_positions = group_centers + STYLE['scatter_offset']
    box_positions = group_centers + STYLE['box_offset']
    violin_positions = group_centers + STYLE['violin_offset']

    box = ax.boxplot(
        data,
        positions=box_positions,
        widths=STYLE['box_width'],
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color='black', linewidth=STYLE['median_width']),
        whiskerprops=dict(color='black', linewidth=STYLE['box_line_width']),
        capprops=dict(color='black', linewidth=STYLE['box_line_width']),
        boxprops=dict(edgecolor='black', linewidth=STYLE['box_line_width']),
    )
    for patch, group in zip(box['boxes'], group_order):
        patch.set_facecolor(lighten_color(colors[group], amount=0.18))
        patch.set_alpha(0.90)

    if show_violin:
        violins = ax.violinplot(
            data,
            positions=violin_positions,
            widths=STYLE['violin_width'],
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for idx, body in enumerate(violins['bodies']):
            group = group_order[idx]
            body.set_facecolor(lighten_color(colors[group], amount=0.30))
            body.set_edgecolor('black')
            body.set_linewidth(STYLE['box_line_width'])
            body.set_alpha(0.70)
            vertices = body.get_paths()[0].vertices
            vertices[:, 0] = np.maximum(vertices[:, 0], violin_positions[idx])

        for pos, values in zip(violin_positions, data):
            if len(values) > 0:
                ax.vlines(pos, np.min(values), np.max(values), color='black', linewidth=1.15, zorder=2)

    if show_scatter:
        rng = np.random.default_rng(2026)
        for pos, values, group in zip(scatter_positions, data, group_order):
            if len(values) == 0:
                continue
            jitter = rng.uniform(-STYLE['point_jitter'], STYLE['point_jitter'], size=len(values))
            ax.scatter(
                pos + jitter,
                values,
                s=STYLE['point_size'],
                color=colors[group],
                alpha=STYLE['point_alpha'],
                edgecolor='white',
                linewidth=0.45,
                zorder=4,
            )

    ax.set_xticks(group_centers)
    ax.set_xticklabels([labels[group] for group in group_order], fontsize=STYLE['tick_size'])
    style_panel_axis(
        ax,
        xlabel='',
        ylabel=ylabel,
        y_ticks=STYLE['score_y_ticks'],
        xlim=(1.0 - STYLE['distribution_left_margin'], STYLE['distribution_start'] + slot_span + 0.26),
        ylim=STYLE['score_ylim'],
    )
    ax.set_xticks(group_centers)
    ax.set_xticklabels([labels[group] for group in group_order], fontsize=STYLE['tick_size'])
    ax.spines['bottom'].set_bounds(group_centers[0], group_centers[-1])


def add_panel_label(ax, label):
    ax.text(
        -0.16,
        1.08,
        label,
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=STYLE['panel_label_size'],
        fontweight='bold',
        clip_on=False,
    )

In [3]:
# Load and prepare Task 1 data
if not TASK1_DATA_PATH.exists():
    raise FileNotFoundError(f'File not found: {TASK1_DATA_PATH}')

task1_df = pd.read_csv(TASK1_DATA_PATH, index_col=0)
task1_df['round'] = pd.to_numeric(task1_df['round'], errors='coerce')
task1_df['score'] = pd.to_numeric(task1_df['score'], errors='coerce')
task1_df = task1_df.dropna(subset=['phase', 'subject', 'round', 'score']).copy()

task1_order = ['P1', 'P2', 'P2-only']
task1_labels = {group: TASK1_LABELS.get(group, group) for group in task1_order}
task1_colors = {group: TASK1_COLORS.get(group, '#7f7f7f') for group in task1_order}
task1_max_trial = {'P1': 30, 'P2': 50, 'P2-only': 50}

task1_learning_df = task1_df[task1_df['phase'].isin(task1_order)].copy()
task1_curves = compute_learning_curves(
    task1_learning_df,
    subject_col='subject',
    group_col='phase',
    trial_col='round',
    score_col='score',
    group_order=task1_order,
    max_trial_by_group=task1_max_trial,
)

task1_score_records = []
for phase in task1_order:
    phase_df = task1_df.loc[task1_df['phase'] == phase].copy()
    window = task1_max_trial[phase]
    for subject, subject_df in phase_df.groupby('subject'):
        window_df = subject_df.sort_values('round').tail(window)
        if not window_df.empty:
            task1_score_records.append({
                'phase': phase,
                'subject': subject,
                'best_score': window_df['score'].max(),
            })

task1_score_df = pd.DataFrame(task1_score_records)
print('Task 1 participants per phase:')
print(task1_score_df.groupby('phase')['subject'].nunique().reindex(task1_order))

Task 1 participants per phase:
phase
P1         51
P2         51
P2-only    54
Name: subject, dtype: int64


In [4]:
# Load and prepare Task 2 data
if not TASK2_DATA_PATH.exists():
    raise FileNotFoundError(f'File not found: {TASK2_DATA_PATH}')

task2_raw = pd.read_csv(TASK2_DATA_PATH)
task2_df = task2_raw.rename(columns={'trial_id': 'trial'}).copy()
task2_df['trial'] = pd.to_numeric(task2_df['trial'], errors='coerce')
task2_df['score_noisy'] = pd.to_numeric(task2_df['score_noisy'], errors='coerce')

task2_group_map = {
    'observation': 'FDS-Obs',
    'non_observation': 'FDS-NoObs',
    'disrupt FDS': 'NoFDS-Obs',
}
task2_df['plot_group'] = task2_df['group'].map(task2_group_map)
task2_df = task2_df.dropna(subset=['subject_id', 'trial', 'score_noisy', 'plot_group']).copy()

task2_order = ['FDS-Obs', 'FDS-NoObs']
task2_df = task2_df[task2_df['plot_group'].isin(task2_order)].copy()
task2_labels = {group: TASK2_LABELS.get(group, group) for group in task2_order}
task2_colors = {group: TASK2_COLORS.get(group, '#7f7f7f') for group in task2_order}
task2_max_trial = {group: 60 for group in task2_order}

# Safety check: each subject should belong to only one plotted group.
group_check = task2_df.groupby('subject_id')['plot_group'].nunique(dropna=True)
bad_subjects = group_check[group_check > 1].index.tolist()
if bad_subjects:
    raise ValueError(f'Some Task 2 subjects appear in more than one group: {bad_subjects[:10]}')

task2_curves = compute_learning_curves(
    task2_df,
    subject_col='subject_id',
    group_col='plot_group',
    trial_col='trial',
    score_col='score_noisy',
    group_order=task2_order,
    max_trial_by_group=task2_max_trial,
)

task2_score_df = (
    task2_df.groupby(['subject_id', 'plot_group'], as_index=False)
    .agg(best_score=('score_noisy', 'max'))
)
print('Task 2 participants per group:')
print(task2_score_df.groupby('plot_group')['subject_id'].nunique().reindex(task2_order))

Task 2 participants per group:
plot_group
FDS-Obs      40
FDS-NoObs    40
Name: subject_id, dtype: int64


In [5]:
# Draw one integrated four-panel figure
fig = plt.figure(figsize=STYLE['combined_figsize'])
gs = fig.add_gridspec(
    1,
    4,
    width_ratios=[1.0, 1.0, 1.0, 1.0],
    wspace=0.55,
)
axes = [fig.add_subplot(gs[0, idx]) for idx in range(4)]

plot_learning_panel(
    axes[0],
    task1_curves,
    group_order=task1_order,
    colors=task1_colors,
    labels=task1_labels,
    ylim=STYLE['task1_learning_ylim'],
    y_ticks=STYLE['task1_learning_y_ticks'],
    show_legend=True,
)
plot_raincloud_panel(
    axes[1],
    task1_score_df,
    group_col='phase',
    score_col='best_score',
    group_order=task1_order,
    colors=task1_colors,
    labels=task1_labels,
    show_violin=False,
    show_scatter=True,
)
plot_learning_panel(
    axes[2],
    task2_curves,
    group_order=task2_order,
    colors=task2_colors,
    labels=task2_labels,
    ylim=STYLE['task2_learning_ylim'],
    y_ticks=STYLE['task2_learning_y_ticks'],
    show_legend=True,
)
plot_raincloud_panel(
    axes[3],
    task2_score_df,
    group_col='plot_group',
    score_col='best_score',
    group_order=task2_order,
    colors=task2_colors,
    labels=task2_labels,
    show_violin=False,
    show_scatter=True,
    group_alignment='left',
)

combined_path = OUTPUT_DIR / 'figure1_integrated_four_panels.png'
combined_pdf_path = OUTPUT_DIR / 'figure1_integrated_four_panels.pdf'
fig.savefig(combined_path, dpi=STYLE['dpi'], bbox_inches='tight')
fig.savefig(combined_pdf_path, bbox_inches='tight')
print(f'Saved integrated PNG to: {combined_path.resolve()}')
print(f'Saved integrated PDF to: {combined_pdf_path.resolve()}')
plt.show()

Saved integrated PNG to: C:\Users\DELL\Desktop\humans-combine-value-learning-and-hypothesis-testing-main\figures_saving_for_layout\figure1_integrated_four_panels.png
Saved integrated PDF to: C:\Users\DELL\Desktop\humans-combine-value-learning-and-hypothesis-testing-main\figures_saving_for_layout\figure1_integrated_four_panels.pdf


C:\Users\DELL\AppData\Local\Temp\ipykernel_16144\3329789042.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Optional: save the four standardized panels separately from the same plotting helpers.
# These are useful when assembling the final layout in another application.
panel_specs = [
    ('figure1_panel_e_task1_learning_curve.png', STYLE['learning_figsize'], lambda ax: plot_learning_panel(
        ax, task1_curves, group_order=task1_order, colors=task1_colors,
        labels=task1_labels, ylim=STYLE['task1_learning_ylim'],
        y_ticks=STYLE['task1_learning_y_ticks'], show_legend=True,
    )),
    ('figure1_panel_e_task1_best_score_distribution.png', STYLE['distribution_figsize'], lambda ax: plot_raincloud_panel(
        ax, task1_score_df, group_col='phase', score_col='best_score',
        group_order=task1_order, colors=task1_colors, labels=task1_labels,
        show_violin=False, show_scatter=True,
    )),
    ('figure1_panel_f_task2_learning_curve.png', STYLE['learning_figsize'], lambda ax: plot_learning_panel(
        ax, task2_curves, group_order=task2_order, colors=task2_colors,
        labels=task2_labels, ylim=STYLE['task2_learning_ylim'],
        y_ticks=STYLE['task2_learning_y_ticks'], show_legend=True,
    )),
    ('figure1_panel_f_task2_best_score_distribution.png', STYLE['distribution_figsize'], lambda ax: plot_raincloud_panel(
        ax, task2_score_df, group_col='plot_group', score_col='best_score',
        group_order=task2_order, colors=task2_colors, labels=task2_labels,
        show_violin=False, show_scatter=True, group_alignment='left',
    )),
]

for filename, figsize, draw_func in panel_specs:
    panel_fig, panel_ax = plt.subplots(figsize=figsize)
    draw_func(panel_ax)
    panel_path = OUTPUT_DIR / filename
    panel_fig.savefig(panel_path, dpi=STYLE['dpi'], bbox_inches='tight')
    plt.close(panel_fig)
    print(f'Saved panel to: {panel_path.resolve()}')

Saved panel to: C:\Users\DELL\Desktop\humans-combine-value-learning-and-hypothesis-testing-main\figures_saving_for_layout\figure1_panel_e_task1_learning_curve.png
Saved panel to: C:\Users\DELL\Desktop\humans-combine-value-learning-and-hypothesis-testing-main\figures_saving_for_layout\figure1_panel_e_task1_best_score_distribution.png


Saved panel to: C:\Users\DELL\Desktop\humans-combine-value-learning-and-hypothesis-testing-main\figures_saving_for_layout\figure1_panel_f_task2_learning_curve.png
Saved panel to: C:\Users\DELL\Desktop\humans-combine-value-learning-and-hypothesis-testing-main\figures_saving_for_layout\figure1_panel_f_task2_best_score_distribution.png
